# Session 3 - Window Functions, CTEs, Subqueries

Welcome. In this session we will learn:

- **subqueries** (SQL inside parentheses)
- **CTEs** (`WITH ... AS`) for readable steps
- **window functions** that keep detail rows

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - Subqueries

A subquery is SQL inside parentheses.

Three common shapes:

1. **Scalar** - returns one value (for example average price)
2. **IN list** - returns many ids
3. **EXISTS** - checks whether at least one matching row exists

Start with the simplest: compare each product to the average price.


### Products priced above the average


In [1]:
SELECT product_id, product_name, unit_price
FROM training.products
WHERE unit_price > (SELECT AVG(unit_price) FROM training.products)
ORDER BY unit_price DESC
LIMIT 20;


+------------+-----------------------------------+------------+
| product_id | product_name                      | unit_price |
+------------+-----------------------------------+------------+
|        239 | Acer PowerEdge R660 / SKU-0239    |  655846.03 |
|        839 | Acer PowerEdge R660 / SKU-0839    |  655585.23 |
|       1199 | Acer PowerEdge R660 / SKU-1199    |  654902.16 |
|        119 | Acer PowerEdge R660 / SKU-0119    |  654400.14 |
|       1439 | Acer PowerEdge R660 / SKU-1439    |  653037.20 |
|        959 | Acer PowerEdge R660 / SKU-0959    |  652572.44 |
|       1079 | Acer PowerEdge R660 / SKU-1079    |  651707.57 |
|        719 | Acer PowerEdge R660 / SKU-0719    |  651526.14 |
|        359 | Acer PowerEdge R660 / SKU-0359    |  651491.61 |
|        479 | Acer PowerEdge R660 / SKU-0479    |  651107.81 |
|       1319 | Acer PowerEdge R660 / SKU-1319    |  650005.15 |
|        599 | Acer PowerEdge R660 / SKU-0599    |  648706.47 |
|        674 | Acer ThinkSystem SR650 / 

### Customers who placed at least one delivered order (IN)


In [2]:
SELECT customer_id, company_name, city, country
FROM training.customers
WHERE customer_id IN (
  SELECT DISTINCT customer_id
  FROM training.orders
  WHERE status = 'delivered'
    AND customer_id IS NOT NULL
)
ORDER BY company_name
LIMIT 20;


+-------------+-------------------------+-----------+---------+
| customer_id | company_name            | city      | country |
+-------------+-------------------------+-----------+---------+
|         384 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         304 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         104 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         184 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         224 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         584 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         784 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         464 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         504 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|          64 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         424 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         264 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         664 | Beacon Networks Pvt Ltd 

### Orders with no payment (NOT EXISTS)


In [3]:
SELECT o.order_id, o.order_date, o.status, o.freight
FROM training.orders o
WHERE NOT EXISTS (
  SELECT 1 FROM training.payments p WHERE p.order_id = o.order_id
)
ORDER BY o.order_date DESC
LIMIT 20;


+----------+------------+-----------+---------+
| order_id | order_date | status    | freight |
+----------+------------+-----------+---------+
|    38499 | 2024-11-30 | delivered |  283.94 |
|    36399 | 2024-11-30 | delivered |  108.53 |
|    34299 | 2024-11-30 | delivered |   98.06 |
|    32199 | 2024-11-30 | delivered |  310.06 |
|    30099 | 2024-11-30 | delivered |  246.08 |
|    27999 | 2024-11-30 | delivered |  348.26 |
|    25899 | 2024-11-30 | delivered |  337.08 |
|    23799 | 2024-11-30 | delivered |  469.15 |
|    21699 | 2024-11-30 | delivered |  435.41 |
|    19599 | 2024-11-30 | delivered |  142.44 |
|    17499 | 2024-11-30 | delivered |  451.12 |
|    15399 | 2024-11-30 | delivered |  108.75 |
|    13299 | 2024-11-30 | delivered |  204.45 |
|    11199 | 2024-11-30 | delivered |  375.58 |
|     9099 | 2024-11-30 | delivered |  377.07 |
|     6999 | 2024-11-30 | delivered |  147.56 |
|     4899 | 2024-11-30 | delivered |  200.48 |
|     2799 | 2024-11-30 | delivered |  1

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - Subqueries</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-subqueries-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-subqueries-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 2 - CTEs (WITH ... AS)

A **CTE** is a temporary named result you can reuse in the same query.

Why people like them:

- Easier to read than nested subqueries
- You can build the answer in clear steps

Picture the flow: step 1 -> step 2 -> final SELECT.


**CTEs = named steps**

| Step | Role |
|------|------|
| 1. `WITH step1 AS (...)` | First named result |
| 2. `step2 AS (...)` | Next step can read `step1` |
| 3. Final `SELECT` | Use the named steps like tables |

A CTE is a temporary named query for one statement - easier to read than nested subqueries.


### Top cities by order count using a CTE


In [4]:
WITH city_orders AS (
  SELECT c.city, c.country, COUNT(*) AS order_count
  FROM training.orders o
  JOIN training.customers c ON c.customer_id = o.customer_id
  GROUP BY c.city, c.country
)
SELECT city, country, order_count
FROM city_orders
WHERE order_count >= 200
ORDER BY order_count DESC
LIMIT 25;


+------------+-----------+-------------+
| city       | country   | order_count |
+------------+-----------+-------------+
| Austin     | USA       |        1958 |
| Chicago    | USA       |        1958 |
| Hyderabad  | India     |        1958 |
| Chennai    | India     |        1958 |
| Pune       | India     |        1958 |
| Dubai      | UAE       |        1958 |
| Abu Dhabi  | UAE       |        1958 |
| Singapore  | Singapore |        1958 |
| Delhi      | India     |        1957 |
| Frankfurt  | Germany   |        1957 |
| Berlin     | Germany   |        1957 |
| London     | UK        |        1957 |
| New York   | USA       |        1957 |
| Jaipur     | India     |        1957 |
| Ahmedabad  | India     |        1957 |
| Kolkata    | India     |        1957 |
| Bengaluru  | India     |        1957 |
| Mumbai     | India     |        1957 |
| Kochi      | India     |        1566 |
| Manchester | UK        |        1566 |
+------------+-----------+-------------+


### Two-step CTE: monthly order volume then rank months


In [5]:
WITH monthly AS (
  SELECT EXTRACT(YEAR FROM order_date) AS order_year,
         EXTRACT(MONTH FROM order_date) AS order_month,
         COUNT(*) AS order_count
  FROM training.orders
  GROUP BY EXTRACT(YEAR FROM order_date), EXTRACT(MONTH FROM order_date)
)
SELECT order_year, order_month, order_count,
       RANK() OVER (ORDER BY order_count DESC) AS volume_rank
FROM monthly
ORDER BY order_year, order_month;


+------------+-------------+-------------+-------------+
| order_year | order_month | order_count | volume_rank |
+------------+-------------+-------------+-------------+
|       2023 |           1 |        1797 |           2 |
|       2023 |           2 |        1624 |          23 |
|       2023 |           3 |        1798 |           1 |
|       2023 |           4 |        1721 |          14 |
|       2023 |           5 |        1767 |           3 |
|       2023 |           6 |        1710 |          15 |
|       2023 |           7 |        1767 |           3 |
|       2023 |           8 |        1767 |           3 |
|       2023 |           9 |        1710 |          15 |
|       2023 |          10 |        1767 |           3 |
|       2023 |          11 |        1710 |          15 |
|       2023 |          12 |        1767 |           3 |
|       2024 |           1 |        1767 |           3 |
|       2024 |           2 |        1653 |          22 |
|       2024 |           3 |   

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - CTEs</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-cte-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-cte-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Window functions

`GROUP BY` collapses many rows into one.

Window functions keep **every row** and add a calculated column beside it.

Core idea:

```sql
function(...) OVER (
  PARTITION BY some_column
  ORDER BY another_column
)
```

- `PARTITION BY` = reset the calculation for each group
- `ORDER BY` inside OVER = ranking / running order


**Window functions keep detail rows**

| Step | What happens |
|------|----------------|
| 1. Rows | Keep every detail row |
| 2. `OVER (...)` | Define partition / order window |
| 3. Function | `ROW_NUMBER` / `RANK` / `SUM` adds a column |

Unlike `GROUP BY`, window functions do **not** collapse rows - they add calculated columns beside the originals.


### Rank products by price inside each category


In [6]:
SELECT c.category_name, p.product_name, p.unit_price,
       RANK() OVER (
         PARTITION BY c.category_name
         ORDER BY p.unit_price DESC
       ) AS price_rank
FROM training.products p
JOIN training.categories c ON c.category_id = p.category_id
ORDER BY c.category_name, price_rank, p.product_name
LIMIT 40;


+---------------+------------------------------------+------------+------------+
| category_name | product_name                       | unit_price | price_rank |
+---------------+------------------------------------+------------+------------+
| Accessories   | LG Kensington Lock / SKU-0637      |    3539.53 |          1 |
| Accessories   | LG Kensington Lock / SKU-0517      |    3537.78 |          2 |
| Accessories   | LG Kensington Lock / SKU-1237      |    3532.18 |          3 |
| Accessories   | LG Kensington Lock / SKU-0757      |    3531.35 |          4 |
| Accessories   | LG Kensington Lock / SKU-0157      |    3529.85 |          5 |
| Accessories   | LG Kensington Lock / SKU-0037      |    3528.05 |          6 |
| Accessories   | LG Kensington Lock / SKU-0997      |    3515.42 |          7 |
| Accessories   | LG Kensington Lock / SKU-1477      |    3506.17 |          8 |
| Accessories   | LG Kensington Lock / SKU-1357      |    3498.22 |          9 |
| Accessories   | LG Kensing

### ROW_NUMBER for each customer's orders


In [7]:
SELECT o.customer_id, o.order_id, o.order_date, o.status,
       ROW_NUMBER() OVER (
         PARTITION BY o.customer_id
         ORDER BY o.order_date, o.order_id
       ) AS order_seq
FROM training.orders o
WHERE o.customer_id IS NOT NULL
ORDER BY o.customer_id, order_seq
LIMIT 40;


+-------------+----------+------------+------------+-----------+
| customer_id | order_id | order_date | status     | order_seq |
+-------------+----------+------------+------------+-----------+
|           1 |        1 | 2023-01-02 | processing |         1 |
|           1 |    33601 | 2023-01-02 | processing |         2 |
|           1 |    28801 | 2023-04-12 | processing |         3 |
|           1 |    24001 | 2023-07-21 | processing |         4 |
|           1 |    19201 | 2023-10-29 | processing |         5 |
|           1 |    14401 | 2024-02-06 | processing |         6 |
|           1 |     9601 | 2024-05-16 | processing |         7 |
|           1 |     4801 | 2024-08-24 | processing |         8 |
|           1 |    38401 | 2024-08-24 | processing |         9 |
|           2 |        2 | 2023-01-03 | shipped    |         1 |
|           2 |    33602 | 2023-01-03 | shipped    |         2 |
|           2 |    28802 | 2023-04-13 | shipped    |         3 |
|           2 |    24002 

### Running freight total by day (sample month)


In [8]:
SELECT order_date, order_id, freight,
       SUM(freight) OVER (
         ORDER BY order_date, order_id
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_freight
FROM training.orders
WHERE order_date BETWEEN DATE '2023-03-01' AND DATE '2023-03-15'
ORDER BY order_date, order_id;


+------------+----------+---------+-----------------+
| order_date | order_id | freight | running_freight |
+------------+----------+---------+-----------------+
| 2023-03-01 |       59 |  455.14 |          455.14 |
| 2023-03-01 |      759 |  279.31 |          734.45 |
| 2023-03-01 |     1459 |  493.26 |         1227.71 |
| 2023-03-01 |     2159 |  146.57 |         1374.28 |
| 2023-03-01 |     2859 |  223.58 |         1597.86 |
| 2023-03-01 |     3559 |  312.69 |         1910.55 |
| 2023-03-01 |     4259 |   91.77 |         2002.32 |
| 2023-03-01 |     4959 |  144.29 |         2146.61 |
| 2023-03-01 |     5659 |  295.67 |         2442.28 |
| 2023-03-01 |     6359 |  166.87 |         2609.15 |
| 2023-03-01 |     7059 |  340.01 |         2949.16 |
| 2023-03-01 |     7759 |  220.38 |         3169.54 |
| 2023-03-01 |     8459 |  495.74 |         3665.28 |
| 2023-03-01 |     9159 |  228.10 |         3893.38 |
| 2023-03-01 |     9859 |  185.28 |         4078.66 |
| 2023-03-01 |    10559 |  3

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - Window functions</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-windows-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-windows-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
